# Análise de Engajamento no Hacker News com BigQuery

## Objetivo

Este projeto tem como objetivo aplicar consultas SQL analíticas no Google BigQuery utilizando uma base pública de dados, explorando padrões de publicação e engajamento no Hacker News.

A análise busca identificar características relacionadas ao desempenho das publicações, considerando métricas como score, quantidade de comentários, autores, domínios, períodos de publicação e evolução do engajamento.

## Ferramentas utilizadas

- Google BigQuery
- Visual Studio Code
- Google Cloud Code
- SQL
- Jupyter Notebook

### Configuração do ambiente

O ambiente de trabalho foi configurado no Visual Studio Code, com suporte do Google Cloud Code para integração com os serviços do Google Cloud.

A conexão com o Google BigQuery foi configurada e validada, permitindo o acesso ao projeto utilizado durante a atividade e à base pública `bigquery-public-data.hacker_news`.

Após a configuração do ambiente e das permissões necessárias, as consultas SQL foram desenvolvidas e testadas no BigQuery, e seus resultados foram utilizados na documentação das análises realizadas neste notebook.

## Base de dados utilizada

Foi utilizada a base pública `bigquery-public-data.hacker_news`, disponibilizada no Google BigQuery, com foco na tabela `full`.

A base foi escolhida por possuir um grande volume de registros históricos do Hacker News e por disponibilizar informações relevantes para análises de engajamento, como tipo da publicação, autor, data e horário, score, quantidade de comentários e URL.

Essas características permitem aplicar filtros, agregações, ordenações, funções de data e cálculos derivados para investigar diferentes padrões de comportamento das publicações.

# 1. Exploração inicial da base de dados

Antes do desenvolvimento das análises, foi realizada uma exploração inicial da base pública `bigquery-public-data.hacker_news`, disponível no Google BigQuery.

A tabela selecionada para a análise foi `full`, que reúne registros históricos de diferentes tipos de conteúdo publicados no Hacker News.

## Estrutura e volume dos dados

A consulta ao esquema da tabela permitiu identificar os campos disponíveis, seus respectivos tipos de dados e a possibilidade de ocorrência de valores nulos.

Entre os principais campos identificados estão:

- `title`: título da publicação — STRING;
- `url`: endereço associado à publicação — STRING;
- `text`: conteúdo textual do registro — STRING;
- `dead`: indicador de registro marcado como inativo — BOOLEAN;
- `by`: autor responsável pela publicação — STRING;
- `score`: pontuação recebida pela publicação — INTEGER;
- `time`: data e horário em formato Unix — INTEGER;
- `timestamp`: data e horário do registro — TIMESTAMP;
- `type`: tipo do conteúdo — STRING;
- `id`: identificador único do registro — INTEGER;
- `parent`: identificador do registro relacionado — INTEGER;
- `descendants`: quantidade de comentários associados à publicação — INTEGER;
- `ranking`: posição associada ao comentário — INTEGER;
- `deleted`: indicador de registro excluído — BOOLEAN.

Os campos da tabela são definidos como `NULLABLE`, indicando que podem apresentar valores nulos dependendo do tipo e das características de cada registro.

Para verificar o volume da base, foi realizada uma contagem dos registros da tabela `full`, que apresentou **49.299.014 registros** no momento da análise.

## Amostra inicial dos registros

Como parte da exploração, foi realizada uma consulta com uma amostra de 100 registros do tipo `story`, selecionando os campos `id`, `title`, `by`, `score`, `timestamp` e `type`.

A inspeção dessa amostra permitiu observar a estrutura dos registros e identificar a presença de valores nulos em algumas variáveis. Essa característica foi considerada posteriormente no desenvolvimento das consultas analíticas, com a aplicação de tratamentos específicos quando necessário.

## Estratégia de exploração

A exploração inicial considerou:

- estrutura e campos disponíveis na tabela;
- tipos de dados;
- volume total de registros;
- tipos de conteúdo existentes;
- presença de valores nulos;
- período histórico disponível;
- identificação das variáveis relevantes para análise de publicação e engajamento.

A partir dessa exploração, foram selecionadas principalmente as variáveis `by`, `score`, `descendants`, `timestamp`, `url` e `type` para o desenvolvimento das consultas analíticas.

## 2. Estrutura das análises

A atividade solicita o desenvolvimento de quatro queries SQL analíticas, que foram desenvolvidas e documentadas como parte obrigatória da entrega.

Além das quatro consultas solicitadas, foram desenvolvidas oito consultas complementares para aprofundar a exploração dos dados e ampliar a análise dos padrões de publicação e engajamento no Hacker News.

Dessa forma, o notebook apresenta **12 queries analíticas**, mantendo as quatro consultas exigidas pela atividade e acrescentando análises complementares.

## 3. Consultas SQL e análise dos resultados

### Query 01 — Amostra de publicações

#### Objetivo

Realizar uma exploração inicial dos registros do tipo `story`, verificando campos como identificador, título, autor, score, data de publicação e tipo do registro.


#### Principais recursos SQL utilizados

- `SELECT`: seleciona os campos relevantes;
- `AS`: renomeia `by` para `autor`;
- `WHERE`: filtra apenas registros do tipo `story`;
- `LIMIT`: restringe o resultado a 100 registros para facilitar a inspeção.

#### Interpretação do resultado

A consulta retornou uma amostra de 100 publicações do tipo `story`.

A inspeção revelou que alguns registros apresentam valores nulos em campos como título, autor e score, mostrando que o tipo `story` não garante o preenchimento de todas as variáveis.

Como a consulta não utiliza `ORDER BY`, os registros servem apenas como amostra exploratória e não representam um ranking ou a distribuição geral das publicações.

#### Código SQL

In [ ]:
SELECT
  id,
  title,
  `by` AS autor,
  score,
  timestamp,
  type
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
LIMIT 100;

### Query 02 — Publicações com alto engajamento

#### Objetivo

Identificar publicações do tipo `story` com score igual ou superior a 500, destacando os conteúdos de maior pontuação e observando também a quantidade de comentários.

#### Principais recursos SQL utilizados

- `SELECT`: seleciona as variáveis relevantes para a análise;
- `AS`: renomeia os campos `title` para `titulo`, `by` para `autor` e `descendants` para `comentarios`, facilitando a interpretação dos resultados;
- `WHERE`: filtra apenas registros do tipo `story`;
- `AND`: aplica o critério de pontuação mínima de 500;
- `ORDER BY`: ordena as publicações pela pontuação, da maior para a menor;
- `LIMIT`: limita o resultado às 100 primeiras publicações.

#### Interpretação do resultado

A consulta identificou as publicações de maior score entre aquelas com pelo menos 500 pontos. O maior resultado foi **“Stephen Hawking has died”**, com **6.015 pontos e 436 comentários**, seguido por **“A Message to Our Customers”**, com **5.771 pontos**, e **“OpenAI's board has fired Sam Altman”**, com **5.710 pontos**.

Os resultados também mostram que **score e quantidade de comentários não são equivalentes**. Por exemplo, a publicação sobre Sam Altman apresentou **2.530 comentários**, mesmo possuindo score inferior ao primeiro colocado. Assim, conteúdos com alta pontuação podem apresentar diferentes níveis de discussão.



#### Código SQL

In [ ]:
SELECT
  id,
  title AS titulo,
  `by` AS autor,
  score,
  descendants AS comentarios,
  timestamp
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND score >= 500
ORDER BY score DESC
LIMIT 100;

### Query 03 — Indicadores gerais de engajamento

#### Objetivo

Calcular indicadores gerais de engajamento das publicações do tipo `story`, criando uma visão consolidada do comportamento da base.

A consulta analisa o volume total de publicações, score médio e mediano, maior score registrado, média de comentários e maior quantidade de comentários observada.

#### Principais recursos SQL utilizados

- `COUNT(*)`: contabiliza o total de publicações analisadas;
- `AVG()`: calcula as médias de score e de comentários;
- `ROUND()`: arredonda as médias para duas casas decimais;
- `MAX()`: identifica os maiores valores de score e de comentários;
- `APPROX_QUANTILES()`: permite obter de forma eficiente uma aproximação da mediana do score em uma base de grande volume;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui da análise os registros em que o campo `score` possui valor nulo.

#### Interpretação do resultado

A consulta analisou **5.982.523 publicações** do tipo `story` com score preenchido. O **score médio foi de 13,70**, enquanto a **mediana foi de apenas 2 pontos**, indicando que a maior parte das publicações apresenta pontuações baixas e que a média é elevada por uma parcela menor de conteúdos com scores muito altos.

O **maior score registrado foi de 6.015 pontos**, resultado consistente com a publicação de maior pontuação identificada na Query 02.

Em relação aos comentários, foram observados **8,04 comentários em média**, enquanto o maior valor registrado chegou a **9.274 comentários**. Esses resultados mostram uma grande diferença entre o comportamento típico das publicações e os casos de maior engajamento.



#### Código SQL

In [ ]:
SELECT
  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  APPROX_QUANTILES(score, 100)[OFFSET(50)] AS mediana_score,
  MAX(score) AS maior_score,
  ROUND(AVG(descendants), 2) AS media_comentarios,
  MAX(descendants) AS maior_numero_comentarios
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND score IS NOT NULL;

### Query 04 — Publicações com maior engajamento

#### Objetivo

Identificar as publicações do tipo `story` com maior quantidade de comentários diretos, relacionando cada publicação aos comentários associados a ela.

#### Principais recursos SQL utilizados

- `AS`: cria aliases para as tabelas e campos utilizados;
- `LEFT JOIN`: relaciona as publicações aos comentários associados;
- `ON`: define a relação entre `s.id` e `c.parent`;
- `AND c.type = 'comment'`: considera apenas registros classificados como comentários;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui publicações sem título;
- `COUNT(c.id)`: contabiliza os comentários diretamente associados a cada publicação;
- `GROUP BY`: agrupa os resultados por publicação;
- `ORDER BY`: ordena as publicações pela quantidade de comentários em ordem decrescente;
- `LIMIT`: restringe o resultado às 100 publicações com maior quantidade de comentários diretos.

#### Interpretação do resultado

A consulta relacionou as publicações aos seus comentários diretos e ordenou os resultados pela quantidade de comentários.

Entre os resultados analisados, **“Ask HN: Share your personal website”** apresentou a maior quantidade, com **2.202 comentários diretos** e score de **950**. Em seguida aparecem **“Ask HN: Could you share your personal blog here?”**, com **1.810 comentários**, e **“Please tell us what features you'd like in news.ycombinator”**, com **1.315 comentários**.

Também foi observada forte presença de publicações do tipo **“Ask HN”**, incluindo várias edições de **“Who is hiring?”**, indicando que conteúdos voltados à participação da comunidade estão entre os que concentram maior volume de respostas diretas.

Os resultados também mostram que quantidade de comentários e `score` não são métricas equivalentes: uma publicação com mais comentários não necessariamente apresenta maior pontuação.

#### Código SQL

In [ ]:
SELECT
  s.id AS story_id,
  s.title AS titulo,
  s.`by` AS autor_story,
  s.score,
  COUNT(c.id) AS total_comentarios
FROM `bigquery-public-data.hacker_news.full` AS s
LEFT JOIN `bigquery-public-data.hacker_news.full` AS c
  ON c.parent = s.id
  AND c.type = 'comment'
WHERE s.type = 'story'
  AND s.title IS NOT NULL
GROUP BY
  story_id,
  titulo,
  autor_story,
  score
ORDER BY total_comentarios DESC
LIMIT 100;

### Query 05 — Engajamento por faixa de score

#### Objetivo

Analisar como o engajamento varia entre diferentes faixas de score, comparando o volume de publicações, o score médio e a quantidade média de comentários.

As publicações foram divididas em cinco faixas: menos de 10 pontos, 10 a 49, 50 a 99, 100 a 499 e 500 pontos ou mais.

#### Principais recursos SQL utilizados

- `CASE WHEN`: classifica as publicações em diferentes faixas de score;
- `COUNT(*)`: contabiliza o total de publicações em cada faixa;
- `AVG()`: calcula a média de score e a média de comentários em cada grupo;
- `ROUND()`: arredonda as médias para duas casas decimais;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui registros sem score ou sem informação de comentários;
- `GROUP BY`: agrupa as publicações de acordo com a faixa de score criada;
- `ORDER BY`: organiza as faixas de score em ordem crescente.

#### Interpretação do resultado

A análise mostrou forte concentração nas faixas de menor score. Foram identificadas **4.241.384 publicações com menos de 10 pontos**, com score médio de **2,30** e média de apenas **0,38 comentário**.

À medida que o score aumenta, cresce também a média de comentários. Publicações entre **100 e 499 pontos** apresentaram média de **114,89 comentários**, enquanto aquelas com **500 pontos ou mais** chegaram a **376,31 comentários em média**.

Ao mesmo tempo, publicações de score elevado são muito menos frequentes: apenas **16.139 registros** aparecem na faixa de 500 pontos ou mais.

Os resultados indicam uma forte associação entre pontuação e volume de discussão, mas não permitem concluir que uma métrica cause diretamente o aumento da outra.



#### Código SQL

In [ ]:
SELECT
  CASE
    WHEN score < 10 THEN '01 - Menos de 10'
    WHEN score < 50 THEN '02 - 10 a 49'
    WHEN score < 100 THEN '03 - 50 a 99'
    WHEN score < 500 THEN '04 - 100 a 499'
    ELSE '05 - 500 ou mais'
  END AS faixa_score,
  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  ROUND(AVG(descendants), 2) AS media_comentarios
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND score IS NOT NULL
  AND descendants IS NOT NULL
GROUP BY faixa_score
ORDER BY faixa_score;

### Query 06 — Engajamento por dia da semana

#### Objetivo

Analisar o comportamento das publicações de acordo com o dia da semana, comparando o volume de publicações, o score médio e a quantidade média de comentários.

A consulta permite verificar diferenças nos níveis de atividade e engajamento entre os sete dias da semana.

#### Principais recursos SQL utilizados

- `EXTRACT(DAYOFWEEK FROM timestamp)`: extrai o número correspondente ao dia da semana a partir da data e horário da publicação;
- `FORMAT_TIMESTAMP()`: transforma o timestamp no nome do dia da semana, facilitando a interpretação dos resultados;
- `COUNT(*)`: contabiliza o total de publicações realizadas em cada dia da semana;
- `AVG()`: calcula o score médio e a média de comentários;
- `ROUND()`: arredonda as médias para duas casas decimais;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui registros sem timestamp ou score;
- `GROUP BY`: agrupa os registros pelo número e pelo nome do dia da semana;
- `ORDER BY`: organiza os resultados seguindo a ordem numérica dos dias da semana.

#### Interpretação do resultado

Os resultados mostram uma diferença entre **volume de publicações e engajamento médio** ao longo da semana.

A **terça-feira apresentou o maior volume**, com **1.020.340 publicações**, mas registrou score médio de **13,12** e média de **7,67 comentários**.

Já o **domingo**, apesar de apresentar o menor volume, com **561.984 publicações**, registrou os maiores indicadores médios: **score de 16,38** e **9,74 comentários por publicação**. O sábado também apresentou valores relativamente elevados, com score médio de **14,82** e **8,61 comentários**.

Assim, os dados indicam que os dias úteis concentram maior atividade de publicação, enquanto o fim de semana apresenta maior engajamento médio. Essa relação representa uma associação observada nos dados e não permite concluir que o dia da semana seja, isoladamente, responsável pelo maior desempenho.



#### Código SQL

In [ ]:
SELECT
  EXTRACT(DAYOFWEEK FROM timestamp) AS numero_dia,
  FORMAT_TIMESTAMP('%A', timestamp) AS dia_semana,
  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  ROUND(AVG(descendants), 2) AS media_comentarios
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND timestamp IS NOT NULL
  AND score IS NOT NULL
GROUP BY numero_dia, dia_semana
ORDER BY numero_dia;

### Query 07 — Engajamento por hora de publicação

#### Objetivo

Analisar o comportamento das publicações de acordo com a hora de publicação, comparando o volume de publicações, o score médio e a quantidade média de comentários ao longo das 24 horas do dia.

#### Principais recursos SQL utilizados

- `EXTRACT(HOUR FROM timestamp)`: extrai a hora da publicação a partir do campo de data e horário;
- `COUNT(*)`: contabiliza o total de publicações em cada hora;
- `AVG()`: calcula o score médio e a média de comentários para cada horário;
- `ROUND()`: arredonda as médias para duas casas decimais;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui registros sem timestamp, score ou informação de comentários;
- `GROUP BY`: agrupa as publicações de acordo com a hora;
- `ORDER BY`: organiza os resultados cronologicamente pelas horas do dia.

#### Interpretação do resultado

A análise identificou diferenças entre o volume de publicações e o engajamento médio ao longo das 24 horas.

O maior volume foi registrado às **16h, com 339.602 publicações**. Entretanto, os maiores indicadores médios ocorreram às **12h**, com **score médio de 17,38** e **9,22 comentários por publicação**.

Já às **07h** foram observados os menores indicadores médios, com **score de 14,75** e **7,17 comentários por publicação**.

Os resultados mostram, portanto, que o horário com maior volume de publicações não corresponde necessariamente ao de maior engajamento médio. Como a consulta considera apenas registros com `timestamp`, `score` e `descendants` preenchidos, valores nulos nessas variáveis não participam dos cálculos. As diferenças observadas representam associações históricas e não permitem concluir que o horário de publicação seja, isoladamente, responsável pelo desempenho.


#### Código SQL

In [ ]:
SELECT
  EXTRACT(HOUR FROM timestamp) AS hora_publicacao,
  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  ROUND(AVG(descendants), 2) AS media_comentarios
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND timestamp IS NOT NULL
  AND score IS NOT NULL
  AND descendants IS NOT NULL
GROUP BY hora_publicacao
ORDER BY hora_publicacao;

### Query 08 — Autores com maior engajamento

#### Objetivo

Identificar autores com histórico consistente de publicações no Hacker News e comparar seu desempenho médio de engajamento.

Para reduzir a influência de resultados isolados, a análise considera somente autores com **pelo menos 50 publicações** do tipo `story`.

#### Principais recursos SQL utilizados

- `COUNT(*)`: contabiliza o total de publicações de cada autor;
- `AVG()`: calcula o score médio e a média de comentários por autor;
- `ROUND()`: arredonda as médias para duas casas decimais;
- `SUM(score)`: calcula o score total acumulado pelas publicações de cada autor;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui registros sem identificação de autor ou sem score;
- `GROUP BY`: agrupa as publicações por autor;
- `HAVING COUNT(*) >= 50`: mantém apenas autores com pelo menos 50 publicações;
- `ORDER BY media_score DESC`: cria o ranking a partir do maior score médio;
- `LIMIT 20`: restringe o resultado aos 20 autores com maior score médio dentro dos critérios estabelecidos.

#### Interpretação do resultado

A consulta identificou os **20 autores com maior score médio** entre aqueles com pelo menos 50 publicações.

O autor `sama` liderou o ranking, com **124 publicações e score médio de 213,32 pontos**, além de média de **98,86 comentários** por publicação.

O autor `whoishiring`, apesar de ocupar a terceira posição em score médio, com **189,63 pontos**, destacou-se pela **maior média de comentários (353,90)** e pelo **maior score acumulado (101.643 pontos)** entre os autores apresentados.

Já `meetpateltech` registrou o **maior volume de publicações**, com **603 registros**, mostrando que diferentes métricas podem produzir perspectivas distintas sobre o desempenho dos autores.

A análise excluiu registros sem autor ou score. Valores nulos em `descendants` não foram filtrados explicitamente e são desconsiderados pelo `AVG()` no cálculo da média. Assim, os resultados reforçam a importância de avaliar conjuntamente volume de publicações, score médio, comentários e desempenho acumulado.

#### Código SQL

In [ ]:
SELECT
  `by` AS autor,
  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  ROUND(AVG(descendants), 2) AS media_comentarios,
  SUM(score) AS score_total
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND `by` IS NOT NULL
  AND score IS NOT NULL
GROUP BY autor
HAVING COUNT(*) >= 50
ORDER BY media_score DESC
LIMIT 20;

### Query 09 — Evolução anual do engajamento

#### Objetivo

Analisar a evolução histórica das publicações e dos indicadores de engajamento no Hacker News ao longo dos anos.

A consulta compara o volume anual de publicações, o score médio e a quantidade média de comentários, permitindo observar mudanças no comportamento da plataforma ao longo do período disponível na base.

#### Principais recursos SQL utilizados

- `EXTRACT(YEAR FROM timestamp)`: extrai o ano de cada publicação a partir do campo de data e horário;
- `COUNT(*)`: contabiliza o total de publicações em cada ano;
- `AVG()`: calcula o score médio e a média de comentários por ano;
- `ROUND()`: arredonda as médias para duas casas decimais;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui registros sem timestamp, score ou informação de comentários;
- `GROUP BY`: agrupa as publicações por ano;
- `ORDER BY`: organiza os resultados em ordem cronológica.

#### Interpretação do resultado

A consulta apresentou a evolução anual das publicações entre **2006 e 2026**, mostrando mudanças no volume de conteúdo e nos indicadores médios de engajamento.

O volume de publicações cresceu significativamente ao longo da série, atingindo seu maior valor em **2020, com 339.630 publicações**. Já os maiores indicadores médios ocorreram posteriormente: **2021 apresentou o maior score médio, com 21,69 pontos**, e **2022 a maior média de comentários, com 12,88 por publicação**.

Em **2025**, foram registradas **301.868 publicações**, com score médio de **19,66 pontos** e média de **10,76 comentários**, valores superiores aos observados no início da série.

Os registros utilizados possuem `timestamp`, `score` e `descendants` preenchidos. Os resultados de **2006** devem ser interpretados com cautela devido ao baixo volume de apenas **47 publicações**, enquanto **2026 representa um período ainda incompleto** e não deve ter seu volume comparado diretamente ao de anos completos.

A análise mostra que o crescimento do volume de publicações e o aumento do engajamento médio não ocorreram necessariamente nos mesmos períodos, descrevendo uma evolução histórica sem estabelecer relação de causalidade.

#### Código SQL

In [ ]:
SELECT
  EXTRACT(YEAR FROM timestamp) AS ano,
  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  ROUND(AVG(descendants), 2) AS media_comentarios
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND timestamp IS NOT NULL
  AND score IS NOT NULL
  AND descendants IS NOT NULL
GROUP BY ano
ORDER BY ano;

### Query 10 — Engajamento por dia da semana e horário

#### Objetivo

Aprofundar a análise temporal do engajamento no Hacker News por meio do cruzamento entre o dia da semana e a hora de publicação.

Enquanto as consultas anteriores analisaram essas dimensões separadamente, esta consulta permite observar combinações específicas de dia e horário, comparando o volume de publicações, o score médio e a quantidade média de comentários.

#### Principais recursos SQL utilizados

- `EXTRACT(DAYOFWEEK FROM timestamp)`: identifica numericamente o dia da semana;
- `FORMAT_TIMESTAMP('%A', timestamp)`: apresenta o nome correspondente ao dia da semana;
- `EXTRACT(HOUR FROM timestamp)`: extrai a hora de publicação;
- `COUNT(*)`: contabiliza o total de publicações em cada combinação de dia e horário;
- `AVG()`: calcula o score médio e a média de comentários;
- `ROUND()`: arredonda as médias para duas casas decimais;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui registros sem timestamp, score ou informação de comentários;
- `GROUP BY`: agrupa os registros pela combinação de dia da semana e hora de publicação;
- `ORDER BY`: organiza os resultados primeiramente pelo dia da semana e, em seguida, pela hora.

#### Interpretação do resultado

A consulta segmentou as publicações simultaneamente por dia da semana e horário, aprofundando as análises realizadas separadamente nas Queries 06 e 07.

No **domingo**, o maior score médio ocorreu às **11h, com 21,66 pontos**, seguido das **12h, com 21,59 pontos**. Esses horários também se destacaram pela média de comentários, chegando a **11,22 comentários por publicação às 11h**. Entretanto, o maior volume do domingo ocorreu às **17h, com 28.728 publicações**.

Na **segunda-feira**, o maior score médio ocorreu à **01h, com 19,18 pontos**, enquanto a maior média de comentários foi registrada às **00h, com 9,46 comentários por publicação**. Já o maior volume ocorreu às **16h, com 55.585 publicações**.

Os resultados reforçam um padrão observado nas análises temporais anteriores: **os períodos com maior volume de publicações não correspondem necessariamente aos períodos com maiores indicadores médios de engajamento**.

A consulta considera apenas registros do tipo `story` com `timestamp`, `score` e `descendants` preenchidos. Como o cruzamento entre dia e horário gera grupos mais específicos, o volume de registros de cada combinação também deve ser considerado na interpretação das médias.

Assim, a Query 10 mostra que o comportamento do engajamento pode variar não apenas entre dias ou horários isoladamente, mas também entre suas combinações. Os resultados representam associações históricas observadas na base e não permitem concluir que determinado dia e horário sejam, isoladamente, responsáveis por gerar maior engajamento.

#### Código SQL

In [ ]:
SELECT
  EXTRACT(DAYOFWEEK FROM timestamp) AS numero_dia,
  FORMAT_TIMESTAMP('%A', timestamp) AS dia_semana,
  EXTRACT(HOUR FROM timestamp) AS hora_publicacao,
  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  ROUND(AVG(descendants), 2) AS media_comentarios
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND timestamp IS NOT NULL
  AND score IS NOT NULL
  AND descendants IS NOT NULL
GROUP BY numero_dia, dia_semana, hora_publicacao
ORDER BY numero_dia, hora_publicacao;

### Query 11 — Desempenho por domínio

#### Objetivo

Analisar o desempenho das publicações de acordo com o domínio associado às URLs compartilhadas, comparando volume de publicações, score médio, média de comentários e score acumulado.

Para tornar a comparação mais consistente, foram considerados apenas domínios com pelo menos **100 publicações**, e o resultado foi limitado aos **30 domínios com maior volume**.

#### Principais recursos SQL utilizados

- `NET.REG_DOMAIN(url)`: extrai o domínio registrável a partir da URL da publicação;
- `COUNT(*)`: contabiliza o total de publicações associadas a cada domínio;
- `AVG()`: calcula o score médio e a média de comentários;
- `ROUND()`: arredonda as médias para duas casas decimais;
- `SUM(score)`: calcula o score total acumulado pelas publicações de cada domínio;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui registros sem URL, score, quantidade de comentários ou domínio identificável;
- `GROUP BY`: agrupa as publicações pelo domínio extraído;
- `HAVING COUNT(*) >= 100`: mantém somente domínios com pelo menos 100 publicações;
- `ORDER BY total_publicacoes DESC`: organiza os domínios do maior para o menor volume de publicações;
- `LIMIT 30`: restringe o resultado aos 30 domínios com maior quantidade de publicações.

#### Interpretação do resultado

O domínio `github.com` apresentou o maior volume, com **216.611 publicações**, além do maior score acumulado, com **4.247.812 pontos**.

Na sequência aparecem `medium.com`, com **117.211 publicações**, e `youtube.com`, com **116.683**. Apesar do alto volume, esses dois domínios apresentaram scores médios menores, de **9,19** e **6,55 pontos**, respectivamente.

A comparação mostra que maior frequência de compartilhamento não significa necessariamente maior desempenho médio. O domínio `github.io`, por exemplo, aparece com **44.557 publicações**, mas apresentou o **maior score médio entre os 30 domínios analisados, com 28,58 pontos**.

Em relação à média de comentários, `reuters.com` apresentou o maior valor, com **17,44 comentários por publicação**, seguido por `bloomberg.com`, com **16,21**, e `bbc.com`, com **15,10**.

Assim, os resultados mostram que volume, score médio, comentários e score acumulado oferecem perspectivas diferentes sobre o desempenho dos domínios. Como a consulta é ordenada por volume de publicações, os 30 resultados representam os domínios mais frequentes dentro dos critérios definidos, e não necessariamente os 30 de maior engajamento.



#### Código SQL

In [ ]:
SELECT
  NET.REG_DOMAIN(url) AS dominio,
  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  ROUND(AVG(descendants), 2) AS media_comentarios,
  SUM(score) AS score_total
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND url IS NOT NULL
  AND score IS NOT NULL
  AND descendants IS NOT NULL
  AND NET.REG_DOMAIN(url) IS NOT NULL
GROUP BY dominio
HAVING COUNT(*) >= 100
ORDER BY total_publicacoes DESC
LIMIT 30;

### Query 12 — Relação entre comentários e score

#### Objetivo

Analisar a relação entre a quantidade de comentários e o score das publicações do Hacker News, agrupando os registros em faixas de comentários e comparando o score médio e mediano de cada grupo.

#### Principais recursos SQL utilizados

- `CASE WHEN`: classifica as publicações em seis faixas de quantidade de comentários;
- `COUNT(*)`: contabiliza o total de publicações em cada faixa;
- `AVG(score)`: calcula o score médio;
- `APPROX_QUANTILES(score, 100)[OFFSET(50)]`: estima a mediana do score;
- `ROUND()`: arredonda o score médio para duas casas decimais;
- `WHERE`: restringe a análise aos registros do tipo `story`;
- `IS NOT NULL`: exclui registros sem score ou informação de comentários;
- `GROUP BY`: agrupa os registros pelas faixas de comentários;
- `ORDER BY`: organiza as faixas em ordem crescente de quantidade de comentários.

#### Interpretação do resultado

A análise mostrou uma associação crescente entre a quantidade de comentários e o score das publicações.

O maior grupo foi o de publicações **sem comentários**, com **3.173.329 registros**, score médio de **2,40 pontos** e mediana de **2 pontos**. Entre as publicações com **1 a 5 comentários**, o score médio aumentou para **5,29 pontos**, mantendo mediana de **2 pontos**.

Nas faixas seguintes, o crescimento tornou-se mais expressivo:

- **6 a 20 comentários:** média de 31,99 e mediana de 22;
- **21 a 50 comentários:** média de 78,80 e mediana de 64;
- **51 a 100 comentários:** média de 128,79 e mediana de 104;
- **mais de 100 comentários:** média de 305,95 e mediana de 235.

A progressão observada tanto na média quanto na mediana indica uma **forte associação entre volume de discussão e pontuação das publicações**. Em todas as faixas, a média permaneceu acima da mediana, indicando influência de publicações com scores elevados na distribuição.

Essa análise complementa a **Query 05**, que observou a relação pela perspectiva inversa, agrupando as publicações por faixa de score e analisando seus comentários.

Os resultados demonstram associação, mas **não permitem estabelecer causalidade**: não é possível concluir que receber mais comentários cause aumento do score ou que um score maior seja responsável pelo aumento dos comentários.



#### Código SQL

In [ ]:
SELECT
  CASE
    WHEN descendants = 0 THEN '01 - Sem comentários'
    WHEN descendants <= 5 THEN '02 - 1 a 5'
    WHEN descendants <= 20 THEN '03 - 6 a 20'
    WHEN descendants <= 50 THEN '04 - 21 a 50'
    WHEN descendants <= 100 THEN '05 - 51 a 100'
    ELSE '06 - Mais de 100'
  END AS faixa_comentarios,

  COUNT(*) AS total_publicacoes,
  ROUND(AVG(score), 2) AS media_score,
  APPROX_QUANTILES(score, 100)[OFFSET(50)] AS mediana_score

FROM `bigquery-public-data.hacker_news.full`

WHERE type = 'story'
  AND score IS NOT NULL
  AND descendants IS NOT NULL

GROUP BY faixa_comentarios
ORDER BY faixa_comentarios;

## 4. Validação dos dados

A validação dos dados considerou consistência dos resultados, presença de valores nulos e possíveis duplicidades.

Durante a exploração inicial, foram identificados valores nulos em diferentes campos da tabela. Por esse motivo, as consultas analíticas aplicaram filtros com `IS NOT NULL` sempre que determinada variável era necessária para os cálculos.

Também foi realizada uma verificação de duplicidades utilizando o campo `id`, identificador único dos registros do Hacker News. A tabela apresentou **49.299.014 registros e 49.299.014 IDs únicos**, resultando em **0 possíveis duplicados**.

Além disso, os resultados das diferentes consultas foram comparados entre si para verificar consistência. As métricas e agrupamentos apresentaram valores compatíveis com os filtros e critérios definidos em cada análise.

In [ ]:
SELECT
  COUNT(*) AS total_registros,
  COUNT(DISTINCT id) AS ids_unicos,
  COUNT(*) - COUNT(DISTINCT id) AS possiveis_duplicados
FROM `bigquery-public-data.hacker_news.full`;

## 5. Conclusão

A análise da base pública do Hacker News permitiu explorar diferentes aspectos relacionados ao volume de publicações e ao engajamento da comunidade ao longo do período disponível.

Os resultados mostraram que a maior parte das publicações apresenta baixo score e poucos comentários, enquanto uma parcela menor concentra níveis significativamente mais altos de interação. Também foi observada uma associação crescente entre quantidade de comentários e score, embora essa relação não permita estabelecer causalidade.

As análises temporais demonstraram que os períodos com maior volume de publicações não correspondem necessariamente aos maiores níveis médios de engajamento. A evolução histórica também revelou mudanças no comportamento da plataforma ao longo dos anos.

As análises por autor e domínio reforçaram essa diferença entre volume e desempenho: maior quantidade de publicações não significa, necessariamente, maior score médio ou maior média de comentários.

De forma geral, as consultas permitiram identificar padrões de publicação e engajamento sob diferentes perspectivas, utilizando recursos de filtragem, agregação, agrupamento, funções temporais e cálculos derivados no BigQuery.

Os resultados foram validados quanto à consistência, presença de valores nulos e possíveis duplicidades, não sendo identificados IDs duplicados na tabela analisada.